# ML-based noise fingerprinting: ablations
**Goal:** identify the noise type (depolarizing / amplitude damping / phase damping / readout error) from measurement counts alone, using 5 small entangled-state circuits simulated with Qiskit Aer.

**How to use:** keep this notebook next to the `datasets/` folder and run the cells from top to bottom (or jump to any ablation after running the *Setup* cells).
Set `QUICK = True` in the first code cell for a fast demo (1 dataset, 1 CV seed). `QUICK = False` uses the paper's full settings.

| Section | Ablation |
|---|---|
| 1 | Main classification (5 models x 5 circuits) |
| 2 | Shots per experiment |
| 3 | Number of rows |
| 4 | Noise-strength range |
| 5 | Circuit family (pool size, leave-one-circuit-out) |
| 6 | Feature set |
| 7 | Hyperparameters (one at a time) |
| 8 | Cross-validation folds |
| 9 | Joint tuning (grid search) |
| A | Optional: regenerate the datasets with Qiskit |

> **Note:** the saved outputs below were produced with `QUICK = True` (1 dataset, 1 CV seed), so numbers differ slightly from the paper's tables, which use `QUICK = False`.

## 0. Setup
Install once: `pip install qiskit qiskit-aer scikit-learn xgboost pandas numpy`

In [1]:
# ---- run mode ----
QUICK = True       # True = fast demo (1 dataset x 1 CV seed, ~7 min); set False for the paper's full settings (3 datasets x 2 CV seeds)

In [ ]:
# ---- shared helpers: circuits, noise models, features, classifiers, cross-validation ----
import warnings
from itertools import product
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

HERE = Path.cwd()
DATA, RES = HERE / "datasets", HERE / "results"
RES.mkdir(exist_ok=True)

DRAWS = [0] if QUICK else [0, 1, 2]     # which of the 3 independent datasets to use
SEEDS = [0] if QUICK else [0, 1]        # cross-validation shuffling seeds

# ---- experiment grid (5 x 4 x 6 x 6 = 720 rows per dataset)
SEVERITIES  = [0.001, 0.005, 0.01, 0.02, 0.03, 0.05]
NOISE_TYPES = ["depolarizing", "amplitude_damping", "phase_damping", "readout_error"]
CIRC_ORDER  = ["bell", "ghz", "w_state", "ghz4", "bell_pairs"]
SHOTS_LIST  = [50, 100, 200, 400, 600, 720, 1024, 2048, 4096, 8192, 16384, 32768]
META = ["circuit", "noise_type", "severity", "trial"]
ENG  = ["tvd", "entropy", "zero_bias", "unexpected_bitstring_rate",
        "symmetric_swap_rate", "parity_violation_rate"]          # 6 engineered features
MODELS = ["RF", "XGB", "SVM", "MLP", "LR"]


# =====================================================================
# 1. Circuits  (each returns the circuit and its ideal outcome distribution)
# =====================================================================
def _bell():
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(2, 2); qc.h(0); qc.cx(0, 1); qc.measure([0, 1], [0, 1])
    return qc, {"00": .5, "11": .5}
def _ghz():
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(3, 3); qc.h(0); qc.cx(0, 1); qc.cx(1, 2); qc.measure([0, 1, 2], [0, 1, 2])
    return qc, {"000": .5, "111": .5}
def _w_state():
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(3, 3); qc.ry(1.9106332362490184, 0); qc.ch(0, 1); qc.cx(1, 2); qc.cx(0, 1); qc.x(0)
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc, {"001": 1/3, "010": 1/3, "100": 1/3}
def _ghz4():
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(4, 4); qc.h(0); qc.cx(0, 1); qc.cx(1, 2); qc.cx(2, 3); qc.measure(range(4), range(4))
    return qc, {"0000": .5, "1111": .5}
def _bell_pairs():
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(4, 4); qc.h(0); qc.cx(0, 1); qc.h(2); qc.cx(2, 3); qc.measure(range(4), range(4))
    return qc, {"0000": .25, "0011": .25, "1100": .25, "1111": .25}

def circuits():
    return {"bell": _bell(), "ghz": _ghz(), "w_state": _w_state(), "ghz4": _ghz4(), "bell_pairs": _bell_pairs()}


# =====================================================================
# 2. Noise models (p = noise strength / severity)
# =====================================================================
def noise_model(kind, p, n_qubits):
    from qiskit_aer.noise import (NoiseModel, depolarizing_error, amplitude_damping_error,
                                  phase_damping_error, ReadoutError)
    nm, one_q, two_q = NoiseModel(), ["h", "x", "id", "ry"], ["cx", "ch"]
    if kind == "depolarizing":
        nm.add_all_qubit_quantum_error(depolarizing_error(p, 1), one_q)
        nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), two_q)
    elif kind in ("amplitude_damping", "phase_damping"):
        e = amplitude_damping_error(p) if kind == "amplitude_damping" else phase_damping_error(p)
        nm.add_all_qubit_quantum_error(e, one_q)
        nm.add_all_qubit_quantum_error(e.tensor(e), two_q)
    else:  # readout_error: each measured bit flips with probability p
        ro = ReadoutError([[1 - p, p], [p, 1 - p]])
        for q in range(n_qubits): nm.add_readout_error(ro, [q])
    return nm


# =====================================================================
# 3. Features computed from the measurement counts only
# =====================================================================
def _probs(counts, n):
    tot = sum(counts.values())
    return {"".join(b): counts.get("".join(b), 0) / tot for b in product("01", repeat=n)}

def features(counts, ideal, n):
    pr = _probs(counts, n); full = {s: ideal.get(s, 0.0) for s in pr}
    tvd = 0.5 * sum(abs(pr[s] - full[s]) for s in pr)                       # distance from ideal
    ent = -sum(p * np.log2(p) for p in pr.values() if p > 0)                # Shannon entropy
    zfrac = lambda d: sum(p * (s.count("0") / n) for s, p in d.items())
    support = set(ideal)
    unexp = sum(p for s, p in pr.items() if s not in support)               # prob. of 'impossible' outcomes
    comp = lambda s: "".join("1" if c == "0" else "0" for c in s)
    seen, bal = set(), 0.0
    for s in pr:
        if s in support or s in seen: continue
        c = comp(s); seen.update([s, c])
        if c in support: continue
        bal += 2 * min(pr[s], pr.get(c, 0.0))
    par = lambda d: sum(p for s, p in d.items() if s.count("1") % 2 == 0)
    pe, ie = par(pr), par(full)
    return {"tvd": tvd, "entropy": ent, "zero_bias": zfrac(pr) - zfrac(full),
            "unexpected_bitstring_rate": unexp,
            "symmetric_swap_rate": bal / unexp if unexp > 1e-9 else 0.0,
            "parity_violation_rate": 0.5 * (abs(pe - ie) + abs((1 - pe) - (1 - ie)))}, pr


# =====================================================================
# 4. Dataset generation 
# =====================================================================
def generate_dataset(shots, draw, trials=6):
    from qiskit import transpile
    from qiskit_aer import AerSimulator
    rows = []
    for ci, cname in enumerate(CIRC_ORDER):
        qc, ideal = circuits()[cname]; n = qc.num_qubits
        for ni, nt in enumerate(NOISE_TYPES):
            for si, sev in enumerate(SEVERITIES):
                sim = AerSimulator(noise_model=noise_model(nt, sev, n)); tqc = transpile(qc, sim)
                for trial in range(trials):
                    seed = (draw * 1000003 + ci * 10007 + ni * 1009 + si * 101 + trial * 7 + shots) % (2**31 - 1)
                    counts = sim.run(tqc, shots=shots, seed_simulator=seed).result().get_counts()
                    f, pr = features(counts, ideal, n)
                    r = {"circuit": cname, "noise_type": nt, "severity": sev, "trial": trial, **f}
                    r.update({f"p_{s}": v for s, v in pr.items()})
                    rows.append(r)
    df = pd.DataFrame(rows)
    pc = sorted([c for c in df.columns if c.startswith("p_")], key=lambda c: (len(c), c))
    return df[META + ENG + pc].fillna(0.0)

def load(shots, draw):
    """Load a ready-made dataset: shots in SHOTS_LIST, draw in {0,1,2}. 720 rows x 38 columns."""
    return pd.read_csv(DATA / f"s{shots}_d{draw}.csv")

def feat_cols(df, kind="all"):
    raw = [c for c in df.columns if c.startswith("p_")]       # raw outcome probabilities
    return {"all": ENG + raw, "eng": ENG, "raw": raw}[kind]


# =====================================================================
# 5. Classifiers + 5-fold cross-validation
# =====================================================================
def make_model(name, **override):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    import xgboost as xgb
    d = {"RF":  dict(n_estimators=300, random_state=0, n_jobs=1),
         "XGB": dict(n_estimators=300, max_depth=5, random_state=0, eval_metric="mlogloss", n_jobs=1, tree_method="hist"),
         "SVM": dict(kernel="rbf", C=5, gamma="scale"),
         "MLP": dict(hidden_layer_sizes=(64, 32), max_iter=800, random_state=0),
         "LR":  dict(max_iter=1000)}[name]
    p = {**d, **override}
    if name == "RF":  return RandomForestClassifier(**p)
    if name == "XGB": return xgb.XGBClassifier(**p)
    if name == "SVM": return make_pipeline(StandardScaler(), SVC(**p))
    if name == "MLP": return make_pipeline(StandardScaler(), MLPClassifier(**p))
    if name == "LR":  return make_pipeline(StandardScaler(), LogisticRegression(**p))

def labels(df):
    return df["noise_type"].map({n: i for i, n in enumerate(NOISE_TYPES)}).values

def cv_correct(df, model="RF", params=None, seed=0, k=5, feats="all"):
    """k-fold stratified CV. Returns a True/False array: was each row classified correctly?"""
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    cols = feats if isinstance(feats, list) else feat_cols(df, feats)
    y = labels(df)
    pred = cross_val_predict(make_model(model, **(params or {})), df[cols].values, y,
                             cv=StratifiedKFold(k, shuffle=True, random_state=seed))
    return pred == y

def cv_acc(df, **kw):
    return float(cv_correct(df, **kw).mean())


# =====================================================================
# 6. Print + save a results table (mean +- std of accuracy for each setting)
# =====================================================================
def report(rows, by, name):
    """rows = list of dicts with an 'acc' key; by = column(s) that define a 'setting'."""
    df = pd.DataFrame(rows)
    g = df.groupby(by, sort=False)["acc"].agg(mean=lambda x: 100 * np.mean(x),
                                              std=lambda x: 100 * np.std(x), n="count")
    print(f"\n=== {name} : accuracy % (mean, std over repeats) ===")
    print(g.round(1).to_string())
    g.round(2).to_csv(RES / f"{name}.csv")
    print(f"[saved results/{name}.csv]")
    return g

## 1. Main classification



In [3]:
rows = []
for m in MODELS:
    for c in CIRC_ORDER:
        for d in DRAWS:
            for sd in SEEDS:
                df = load(2048, d).query("circuit == @c").reset_index(drop=True)
                rows.append(dict(model=m, circuit=c, acc=cv_acc(df, model=m, seed=sd)))
report(rows, ["model", "circuit"], "01_main_by_model_and_circuit")
report(rows, ["model"], "01_main_by_model")


=== 01_main_by_model_and_circuit : accuracy % (mean, std over repeats) ===
                  mean  std  n
model circuit                 
RF    bell        58.3  0.0  1
      ghz         86.1  0.0  1
      w_state     86.1  0.0  1
      ghz4        95.1  0.0  1
      bell_pairs  77.8  0.0  1
XGB   bell        61.8  0.0  1
      ghz         84.7  0.0  1
      w_state     83.3  0.0  1
      ghz4        96.5  0.0  1
      bell_pairs  72.2  0.0  1
SVM   bell        66.0  0.0  1
      ghz         81.9  0.0  1
      w_state     84.0  0.0  1
      ghz4        91.0  0.0  1
      bell_pairs  72.9  0.0  1
MLP   bell        59.0  0.0  1
      ghz         82.6  0.0  1
      w_state     84.7  0.0  1
      ghz4        93.8  0.0  1
      bell_pairs  65.3  0.0  1
LR    bell        64.6  0.0  1
      ghz         84.0  0.0  1
      w_state     84.0  0.0  1
      ghz4        91.7  0.0  1
      bell_pairs  66.7  0.0  1
[saved results/01_main_by_model_and_circuit.csv]

=== 01_main_by_model : accuracy % (me

,mean,std,n
model,,,
RF,80.694444,12.456715,5
XGB,79.722222,11.812906,5
SVM,79.166667,8.762117,5
MLP,77.083333,12.902471,5
LR,78.194444,10.655593,5


## 2. Ablation: shots per experiment
```
ABLATION 1 - SHOTS PER EXPERIMENT.
```

In [4]:
rows, by_sev = [], []
for s in SHOTS_LIST:
    for d in DRAWS:
        for sd in SEEDS:
            df = load(s, d)
            for m in MODELS:
                ok = cv_correct(df, model=m, seed=sd)
                rows.append(dict(shots=s, model=m, acc=ok.mean()))
                if m == "RF":                                    # accuracy by noise strength (RF only)
                    for sev, v in df.assign(_ok=ok).groupby("severity")["_ok"].mean().items():
                        by_sev.append(dict(shots=s, severity=sev, acc=v))
report(rows, ["shots", "model"], "02_shots_by_model")
report(by_sev, ["shots", "severity"], "02_shots_rf_by_severity")


=== 02_shots_by_model : accuracy % (mean, std over repeats) ===
             mean  std  n
shots model              
50    RF     42.1  0.0  1
      XGB    39.3  0.0  1
      SVM    44.7  0.0  1
      MLP    39.0  0.0  1
      LR     43.6  0.0  1
100   RF     50.3  0.0  1
      XGB    50.4  0.0  1
      SVM    51.2  0.0  1
      MLP    49.4  0.0  1
      LR     49.9  0.0  1
200   RF     56.4  0.0  1
      XGB    55.7  0.0  1
      SVM    58.5  0.0  1
      MLP    57.1  0.0  1
      LR     56.4  0.0  1
400   RF     67.6  0.0  1
      XGB    64.9  0.0  1
      SVM    62.5  0.0  1
      MLP    63.9  0.0  1
      LR     60.7  0.0  1
600   RF     68.9  0.0  1
      XGB    68.3  0.0  1
      SVM    68.2  0.0  1
      MLP    67.2  0.0  1
      LR     62.9  0.0  1
720   RF     72.9  0.0  1
      XGB    71.8  0.0  1
      SVM    69.4  0.0  1
      MLP    70.0  0.0  1
      LR     65.4  0.0  1
1024  RF     73.8  0.0  1
      XGB    73.6  0.0  1
      SVM    71.8  0.0  1
      MLP    72.2  0.0  1

mean  std  n
shots severity                   
50    0.001     25.833333  0.0  1
      0.005     33.333333  0.0  1
      0.010     35.000000  0.0  1
      0.020     47.500000  0.0  1
      0.030     51.666667  0.0  1
...                   ...  ... ..
32768 0.005     84.166667  0.0  1
      0.010     97.500000  0.0  1
      0.020     95.833333  0.0  1
      0.030     99.166667  0.0  1
      0.050     99.166667  0.0  1

[72 rows x 3 columns]

## 3. Ablation: number of rows
```
ABLATION 2 - NUMBER OF ROWS (experiments used for training).
pool the 3 datasets at 2048 shots (3 x 720 = 2160 rows). For each size n, draw a random sample of n rows that keeps the circuit x noise-type mix equal (stratified), repeat 3 times (different random samples), and run 5-fold CV. n = 2160 is the whole pool.
```

In [5]:
pool = pd.concat([load(2048, d) for d in [0, 1, 2]], ignore_index=True)   # always all 3 draws -> 2160 rows

def stratified_sample(df, n, seed):
    per_group = n // 20                                  # 20 groups = 5 circuits x 4 noise types
    return pd.concat([g.sample(per_group, random_state=seed)
                      for _, g in df.groupby(["circuit", "noise_type"])]).reset_index(drop=True)

SIZES = [100, 200, 300, 400, 500, 600, 720, 1080, 1440, 2160]
rows = []
for rep in range(1 if QUICK else 3):
    for n in SIZES:
        df = pool if n == 2160 else stratified_sample(pool, n, rep)
        for m in MODELS:
            rows.append(dict(n_rows=n, model=m, acc=cv_acc(df, model=m, seed=rep)))
report(rows, ["n_rows", "model"], "03_rows")


=== 03_rows : accuracy % (mean, std over repeats) ===
              mean  std  n
n_rows model              
100    RF     49.0  0.0  1
       XGB    48.0  0.0  1
       SVM    34.0  0.0  1
       MLP    45.0  0.0  1
       LR     45.0  0.0  1
200    RF     67.5  0.0  1
       XGB    63.0  0.0  1
       SVM    58.5  0.0  1
       MLP    70.5  0.0  1
       LR     61.0  0.0  1
300    RF     68.0  0.0  1
       XGB    70.0  0.0  1
       SVM    63.0  0.0  1
       MLP    71.0  0.0  1
       LR     62.0  0.0  1
400    RF     70.2  0.0  1
       XGB    72.0  0.0  1
       SVM    66.5  0.0  1
       MLP    70.8  0.0  1
       LR     62.5  0.0  1
500    RF     74.2  0.0  1
       XGB    74.8  0.0  1
       SVM    73.4  0.0  1
       MLP    76.4  0.0  1
       LR     69.4  0.0  1
600    RF     74.5  0.0  1
       XGB    77.0  0.0  1
       SVM    74.8  0.0  1
       MLP    76.3  0.0  1
       LR     69.0  0.0  1
720    RF     78.6  0.0  1
       XGB    79.3  0.0  1
       SVM    76.8  0.0  1


mean  std  n
n_rows model                   
100    RF     49.000000  0.0  1
       XGB    48.000000  0.0  1
       SVM    34.000000  0.0  1
       MLP    45.000000  0.0  1
       LR     45.000000  0.0  1
200    RF     67.500000  0.0  1
       XGB    63.000000  0.0  1
       SVM    58.500000  0.0  1
       MLP    70.500000  0.0  1
       LR     61.000000  0.0  1
300    RF     68.000000  0.0  1
       XGB    70.000000  0.0  1
       SVM    63.000000  0.0  1
       MLP    71.000000  0.0  1
       LR     62.000000  0.0  1
400    RF     70.250000  0.0  1
       XGB    72.000000  0.0  1
       SVM    66.500000  0.0  1
       MLP    70.750000  0.0  1
       LR     62.500000  0.0  1
500    RF     74.200000  0.0  1
       XGB    74.800000  0.0  1
       SVM    73.400000  0.0  1
       MLP    76.400000  0.0  1
       LR     69.400000  0.0  1
600    RF     74.500000  0.0  1
       XGB    77.000000  0.0  1
       SVM    74.833333  0.0  1
       MLP    76.333333  0.0  1
       LR     69.000000  0.0  1
720    RF     78.611111  0.0  1
       XGB    79.305556  0.0  1
       SVM    76.805556  0.0  1
       MLP    77.361111  0.0  1
       LR     71.527778  0.0  1
1080   RF     77.222222  0.0  1
       XGB    78.981481  0.0  1
       SVM    74.629630  0.0  1
       MLP    76.574074  0.0  1
       LR     69.259259  0.0  1
1440   RF     83.125000  0.0  1
       XGB    81.666667  0.0  1
       SVM    78.055556  0.0  1
       MLP    79.305556  0.0  1
       LR     72.638889  0.0  1
2160   RF     84.583333  0.0  1
       XGB    83.287037  0.0  1
       SVM    78.981481  0.0  1
       MLP    81.620370  0.0  1
       LR     73.703704  0.0  1

## 4. Ablation: noise-strength range
```
ABLATION 3a - NOISE-STRENGTH RANGE.
Method  : keep only rows with severity >= threshold (drop the weakest levels), RF, 2048 shots.
```

In [6]:
rows = []
for th in SEVERITIES[:-1]:                               # 0.001, 0.005, 0.01, 0.02, 0.03
    for d in DRAWS:
        for sd in SEEDS:
            df = load(2048, d).query("severity >= @th").reset_index(drop=True)
            rows.append(dict(min_severity=th, n_rows=len(df), acc=cv_acc(df, model="RF", seed=sd)))
report(rows, ["min_severity", "n_rows"], "04_severity_range")


=== 04_severity_range : accuracy % (mean, std over repeats) ===
                     mean  std  n
min_severity n_rows              
0.001        720     79.4  0.0  1
0.005        600     83.8  0.0  1
0.010        480     88.8  0.0  1
0.020        360     96.7  0.0  1
0.030        240     97.9  0.0  1
[saved results/04_severity_range.csv]


,,mean,std,n
min_severity,n_rows,,,
0.001,720,79.444444,0.0,1
0.005,600,83.833333,0.0,1
0.010,480,88.750000,0.0,1
0.020,360,96.666667,0.0,1
0.030,240,97.916667,0.0,1


## 5. Ablation: circuit family
```
ABLATION 3b - CIRCUIT FAMILY.
(a) Pool size : train+test on the first k circuits only (k = 1..5)  [bell, ghz, w_state, ghz4, bell_pairs].
(b) Transfer  : leave-one-circuit-out (LOCO) - train on 4 circuits, test on the unseen 5th.
RF, 2048 shots.
```

In [7]:
rows = []
for k in range(1, 6):
    for d in DRAWS:
        for sd in SEEDS:
            df = load(2048, d)[lambda x: x.circuit.isin(CIRC_ORDER[:k])].reset_index(drop=True)
            rows.append(dict(n_circuits=k, acc=cv_acc(df, model="RF", seed=sd)))
report(rows, ["n_circuits"], "05a_circuit_pool_size")

loco = []
for held_out in CIRC_ORDER:
    for kind in ["all", "eng"]:                          # all 34 features vs 6 engineered features
        for d in DRAWS:
            df = load(2048, d); cols = feat_cols(df, kind)
            train, test = df[df.circuit != held_out], df[df.circuit == held_out]
            model = make_model("RF").fit(train[cols].values, labels(train))
            acc = float((model.predict(test[cols].values) == labels(test)).mean())
            loco.append(dict(held_out=held_out, features=kind, acc=acc))
report(loco, ["held_out", "features"], "05b_leave_one_circuit_out")


=== 05a_circuit_pool_size : accuracy % (mean, std over repeats) ===
            mean  std  n
n_circuits              
1           58.3  0.0  1
2           72.9  0.0  1
3           78.5  0.0  1
4           81.4  0.0  1
5           79.4  0.0  1
[saved results/05a_circuit_pool_size.csv]



=== 05b_leave_one_circuit_out : accuracy % (mean, std over repeats) ===
                     mean  std  n
held_out   features              
bell       all       56.9  0.0  1
           eng       63.9  0.0  1
ghz        all       49.3  0.0  1
           eng       63.2  0.0  1
w_state    all       20.8  0.0  1
           eng       40.3  0.0  1
ghz4       all       71.5  0.0  1
           eng       75.7  0.0  1
bell_pairs all       58.3  0.0  1
           eng       70.1  0.0  1
[saved results/05b_leave_one_circuit_out.csv]


mean  std  n
held_out   features                   
bell       all       56.944444  0.0  1
           eng       63.888889  0.0  1
ghz        all       49.305556  0.0  1
           eng       63.194444  0.0  1
w_state    all       20.833333  0.0  1
           eng       40.277778  0.0  1
ghz4       all       71.527778  0.0  1
           eng       75.694444  0.0  1
bell_pairs all       58.333333  0.0  1
           eng       70.138889  0.0  1

## 6. Ablation: feature set
```
ABLATION 4 - FEATURE SET
```

In [8]:
raw = feat_cols(load(2048, 0), "raw")
settings = {"all": ENG + raw, "engineered_only": ENG, "raw_only": raw}
for f in ENG: settings[f"drop_{f}"] = [c for c in ENG + raw if c != f]
for f in ENG: settings[f"only_{f}"] = [f]
rows = []
for name, cols in settings.items():
    for d in DRAWS:
        for sd in SEEDS:
            rows.append(dict(feature_set=name, acc=cv_acc(load(2048, d), model="RF", seed=sd, feats=cols)))
report(rows, ["feature_set"], "06_feature_set")


=== 06_feature_set : accuracy % (mean, std over repeats) ===
                                mean  std  n
feature_set                                 
all                             79.4  0.0  1
engineered_only                 73.8  0.0  1
raw_only                        78.8  0.0  1
drop_tvd                        79.3  0.0  1
drop_entropy                    79.2  0.0  1
drop_zero_bias                  78.9  0.0  1
drop_unexpected_bitstring_rate  78.8  0.0  1
drop_symmetric_swap_rate        79.4  0.0  1
drop_parity_violation_rate      78.9  0.0  1
only_tvd                        37.9  0.0  1
only_entropy                    52.9  0.0  1
only_zero_bias                  36.7  0.0  1
only_unexpected_bitstring_rate  51.1  0.0  1
only_symmetric_swap_rate        52.4  0.0  1
only_parity_violation_rate      49.6  0.0  1
[saved results/06_feature_set.csv]


,mean,std,n
feature_set,,,
all,79.444444,0.0,1
engineered_only,73.750000,0.0,1
raw_only,78.750000,0.0,1
drop_tvd,79.305556,0.0,1
drop_entropy,79.166667,0.0,1
drop_zero_bias,78.888889,0.0,1
drop_unexpected_bitstring_rate,78.750000,0.0,1
drop_symmetric_swap_rate,79.444444,0.0,1
drop_parity_violation_rate,78.888889,0.0,1


## 7. Ablation: classifier hyperparameters
```
ABLATION 5 - CLASSIFIER HYPERPARAMETERS (one at a time)
```

In [9]:
HP = {
 "RF":  {"n_estimators": [10, 25, 50, 100, 200, 300, 500], "max_depth": [2, 4, 6, 8, 12, None],
         "max_features": ["sqrt", "log2", 0.5, 1.0], "min_samples_split": [2, 4, 8, 16],
         "min_samples_leaf": [1, 2, 4, 8], "criterion": ["gini", "entropy"], "bootstrap": [True, False]},
 "XGB": {"n_estimators": [50, 100, 200, 300, 500], "max_depth": [2, 3, 5, 7, 9],
         "learning_rate": [0.01, 0.03, 0.1, 0.3, 0.5], "subsample": [0.5, 0.8, 1.0], "colsample_bytree": [0.5, 0.8, 1.0]},
 "SVM": {"C": [0.1, 1, 5, 10, 50, 100], "gamma": ["scale", 0.001, 0.01, 0.1, 1], "kernel": ["linear", "poly", "rbf", "sigmoid"]},
 "MLP": {"hidden_layer_sizes": [(16,), (32,), (64,), (64, 32), (128, 64), (128, 64, 32)], "alpha": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
         "learning_rate_init": [1e-4, 1e-3, 1e-2], "activation": ["relu", "tanh", "logistic"]},
 "LR":  {"C": [0.001, 0.01, 0.1, 1, 10, 100]},
}
rows = []
for model, params in HP.items():
    for name, values in params.items():
        for v in values:
            for d in DRAWS:
                rows.append(dict(model=model, hyperparameter=name, value=str(v),
                                 acc=cv_acc(load(2048, d), model=model, params={name: v}, seed=0)))
report(rows, ["model", "hyperparameter", "value"], "07_hyperparameters")


=== 07_hyperparameters : accuracy % (mean, std over repeats) ===
                                        mean  std  n
model hyperparameter     value                      
RF    n_estimators       10             78.3  0.0  1
                         25             79.6  0.0  1
                         50             81.0  0.0  1
                         100            80.0  0.0  1
                         200            80.1  0.0  1
                         300            79.4  0.0  1
                         500            79.2  0.0  1
      max_depth          2              66.0  0.0  1
                         4              73.8  0.0  1
                         6              77.5  0.0  1
                         8              78.9  0.0  1
                         12             79.9  0.0  1
                         None           79.4  0.0  1
      max_features       sqrt           79.4  0.0  1
                         log2           79.4  0.0  1
                         0.5     

mean  std  n
model hyperparameter value                   
RF    n_estimators   10     78.333333  0.0  1
                     25     79.583333  0.0  1
                     50     80.972222  0.0  1
                     100    80.000000  0.0  1
                     200    80.138889  0.0  1
...                               ...  ... ..
LR    C              0.01   65.138889  0.0  1
                     0.1    70.138889  0.0  1
                     1      71.527778  0.0  1
                     10     71.805556  0.0  1
                     100    71.944444  0.0  1

[88 rows x 3 columns]

## 8. Ablation: cross-validation folds
```
ABLATION 6a - NUMBER OF CROSS-VALIDATION FOLDS (k = 3, 5, 10).
```

In [10]:
rows = []
for k in [3, 5, 10]:
    for m in MODELS:
        for d in DRAWS:
            rows.append(dict(folds=k, model=m, acc=cv_acc(load(2048, d), model=m, seed=0, k=k)))
report(rows, ["folds", "model"], "08_cv_folds")


=== 08_cv_folds : accuracy % (mean, std over repeats) ===
             mean  std  n
folds model              
3     RF     77.6  0.0  1
      XGB    78.6  0.0  1
      SVM    76.1  0.0  1
      MLP    76.4  0.0  1
      LR     71.9  0.0  1
5     RF     79.4  0.0  1
      XGB    79.2  0.0  1
      SVM    76.4  0.0  1
      MLP    77.8  0.0  1
      LR     71.5  0.0  1
10    RF     79.3  0.0  1
      XGB    79.9  0.0  1
      SVM    77.1  0.0  1
      MLP    77.1  0.0  1
      LR     72.4  0.0  1
[saved results/08_cv_folds.csv]


mean  std  n
folds model                   
3     RF     77.638889  0.0  1
      XGB    78.611111  0.0  1
      SVM    76.111111  0.0  1
      MLP    76.388889  0.0  1
      LR     71.944444  0.0  1
5     RF     79.444444  0.0  1
      XGB    79.166667  0.0  1
      SVM    76.388889  0.0  1
      MLP    77.777778  0.0  1
      LR     71.527778  0.0  1
10    RF     79.305556  0.0  1
      XGB    79.861111  0.0  1
      SVM    77.083333  0.0  1
      MLP    77.083333  0.0  1
      LR     72.361111  0.0  1

## 9. Ablation: joint tuning


In [11]:
GRID = {
 "RF":  {"n_estimators": [100, 300], "max_depth": [8, None], "max_features": ["sqrt", 0.5], "min_samples_leaf": [1, 2]},
 "XGB": {"max_depth": [3, 5, 7], "learning_rate": [0.1, 0.3], "n_estimators": [100, 300]},
 "SVM": {"C": [1, 5, 10, 50], "gamma": ["scale", 0.01, 0.1]},
 "MLP": {"hidden_layer_sizes": [(64,), (64, 32), (128, 64)], "alpha": [1e-4, 1e-2]},
 "LR":  {"C": [0.01, 0.1, 1, 10, 100]},
}
RUNS = [(0, [0])] if QUICK else [(0, [0]), (1, [0]), (2, [0, 1, 2])]
rows = []
for model, grid in GRID.items():
    keys = list(grid)
    for combo in product(*[grid[k] for k in keys]):
        params = dict(zip(keys, combo))
        for d, seeds in RUNS:
            for sd in seeds:
                rows.append(dict(model=model, params=str(params), acc=cv_acc(load(2048, d), model=model, params=params, seed=sd)))
g = report(rows, ["model", "params"], "09_joint_tuning_all_combinations")
best = g.reset_index().sort_values("mean", ascending=False).groupby("model").head(1)
print("\nBest grid setting per model:\n", best.to_string(index=False))


=== 09_joint_tuning_all_combinations : accuracy % (mean, std over repeats) ===
                                                                                               mean  std  n
model params                                                                                               
RF    {'n_estimators': 100, 'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 1}     78.9  0.0  1
      {'n_estimators': 100, 'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 2}     79.6  0.0  1
      {'n_estimators': 100, 'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 1}        76.2  0.0  1
      {'n_estimators': 100, 'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 2}        77.5  0.0  1
      {'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1}  80.0  0.0  1
      {'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2}  79.7  0.0  1
      {'n_estimators': 100, 'max_depth': None, 'max_feat

##dataset

In [12]:
REGENERATE = False
if REGENERATE:
    DATA.mkdir(exist_ok=True)
    for draw in range(3):
        for shots in SHOTS_LIST:
            f = DATA / f"s{shots}_d{draw}.csv"
            if f.exists(): continue
            generate_dataset(shots, draw).to_csv(f, index=False); print("wrote", f.name)